### Example 2: Performance Monitoring Agent

Objective: Automatically evaluate a player's injury risk by comparing live data against safety thresholds.
    
    The Scenario: A coach provides a Player ID and a Target Speed. 

    The agent must:Calculate Fatigue: Compare current heart rate against a "Safe Zone" ($Max\ HR \times 0.90$).
    
    Calculate Performance Gap: Determine if the player is hitting the "Target Speed" provided by the coach or if they are lagging.

#### Import the desired libraries and create the model instance

In [20]:
import random
from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool
from langchain.agents import create_agent

# 1. Setup AWS Bedrock (Cohere)

from langchain_aws import ChatBedrockConverse
model=ChatBedrockConverse(model="amazon.nova-lite-v1:0", #anthropic.claude-3-sonnet-20240229-v1:0", #'cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
model.invoke("Hi")

AIMessage(content="Hello! How can I assist you today? If you have any questions or need information on a particular topic, feel free to ask. Whether it's about science, technology, history, or something else, I'm here to help.", additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': 'a05af096-95eb-4e4e-83ca-714dbdc7309d', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Wed, 18 Mar 2026 09:39:50 GMT', 'content-type': 'application/json', 'content-length': '408', 'connection': 'keep-alive', 'x-amzn-requestid': 'a05af096-95eb-4e4e-83ca-714dbdc7309d'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [528]}, 'model_provider': 'bedrock_converse', 'model_name': 'amazon.nova-lite-v1:0'}, id='lc_run--019d0050-cfbb-7a41-8408-11828cc65d1d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1, 'output_tokens': 50, 'total_tokens': 51, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}})

In [21]:
# 2. Tool 1: Dynamic Heart Rate Analysis
@tool
def analyze_cardio_strain(player_id: str, current_hr: int):
    """Calculates if a player's HR is exceeding the 90% safety threshold (200bpm max)."""
    max_hr = 200
    threshold = max_hr * 0.90  # 180 bpm
    
    strain_level = "CRITICAL" if current_hr > threshold else "STABLE"
    diff = current_hr - threshold
    
    return {
        "player": player_id,
        "status": strain_level,
        "bpm_over_threshold": max(0, diff),
        "recommendation": "Immediate rest required" if strain_level == "CRITICAL" else "Continue play"
    }

In [22]:
# 3. Tool 2: Dynamic Speed Efficiency
@tool
def calculate_speed_efficiency(player_id: str, current_speed: float, target_speed: float):
    """Calculates the percentage efficiency vs the coach's target speed."""
    # Efficiency Formula: (Current / Target) * 100
    efficiency = (current_speed / target_speed) * 100
    
    return {
        "player": player_id,
        "efficiency_score": f"{round(efficiency, 1)}%",
        "is_underperforming": efficiency < 85.0,
        "gap": f"{round(target_speed - current_speed, 1)} km/h"
    }

tools = [analyze_cardio_strain, calculate_speed_efficiency]


In [10]:
# 4. Create the Stateless Agent
agent = create_agent(model, tools=tools, system_prompt="You are a helpful Performance Threshold monitor agent for soccer players.")

# 5. The user provides the raw data; the tools do the math.
user_query = {
    "messages": [
        {
            "role": "user", 
            "content": "Analyze Player #18. Their HR is 188 bpm. Their current speed is 24 km/h but I need them hitting 30 km/h. Give me a status report."
        }
    ]
}

response = agent.invoke(user_query)


In [11]:
from rich import print
print(response)

{
    'messages': [
        HumanMessage(
            content='Analyze Player #18. Their HR is 188 bpm. Their current speed is 24 km/h but I need them 
hitting 30 km/h. Give me a status report.',
            additional_kwargs={},
            response_metadata={},
            id='a0eedeea-1530-42fd-b13b-ba7c141d9deb'
        ),
        AIMessage(
            content=[
                {
                    'type': 'text',
                    'text': "<thinking> I need to analyze Player #18's heart rate to see if it is within safe 
limits, and also calculate the efficiency of their current speed compared to the target speed. </thinking>\n"
                },
                {
                    'type': 'tool_use',
                    'name': 'analyze_cardio_strain',
                    'input': {'player_id': '18', 'current_hr': 188},
                    'id': 'tooluse_XX6E8Ftwlj9exFUjpYg6GD'
                },
                {
                    'type': 'tool_use',
                    'name': 'calculate_speed_efficiency',
                    'input': {'current_speed': 24, 'player_id': '18', 'target_speed': 30},
                    'id': 'tooluse_gFd9Tf7hUGyeCfCXJKY6FB'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'ResponseMetadata': {
                    'RequestId': '6933d6fd-cd6a-4ed0-88e7-9d3d9bd2b5f8',
                    'HTTPStatusCode': 200,
                    'HTTPHeaders': {
                        'date': 'Wed, 18 Mar 2026 09:22:03 GMT',
                        'content-type': 'application/json',
                        'content-length': '687',
                        'connection': 'keep-alive',
                        'x-amzn-requestid': '6933d6fd-cd6a-4ed0-88e7-9d3d9bd2b5f8'
                    },
                    'RetryAttempts': 0
                },
                'stopReason': 'tool_use',
                'metrics': {'latencyMs': [969]},
                'model_provider': 'bedrock_converse',
                'model_name': 'amazon.nova-lite-v1:0'
            },
            id='lc_run--019d0040-8872-7d11-a52d-a91c8c630370-0',
            tool_calls=[
                {
                    'name': 'analyze_cardio_strain',
                    'args': {'player_id': '18', 'current_hr': 188},
                    'id': 'tooluse_XX6E8Ftwlj9exFUjpYg6GD',
                    'type': 'tool_call'
                },
                {
                    'name': 'calculate_speed_efficiency',
                    'args': {'current_speed': 24, 'player_id': '18', 'target_speed': 30},
                    'id': 'tooluse_gFd9Tf7hUGyeCfCXJKY6FB',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 570,
                'output_tokens': 100,
                'total_tokens': 670,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0}
            }
        ),
        ToolMessage(
            content='{"player": "18", "status": "CRITICAL", "bpm_over_threshold": 8.0, "recommendation": "Immediate
rest required"}',
            name='analyze_cardio_strain',
            id='7ea5dc06-7769-420c-869e-a33fdb3a533a',
            tool_call_id='tooluse_XX6E8Ftwlj9exFUjpYg6GD'
        ),
        ToolMessage(
            content='{"player": "18", "efficiency_score": "80.0%", "is_underperforming": true, "gap": "6.0 km/h"}',
            name='calculate_speed_efficiency',
            id='dc79dede-fd91-4c62-9621-c7fc880f5ba7',
            tool_call_id='tooluse_gFd9Tf7hUGyeCfCXJKY6FB'
        ),
        AIMessage(
            content="<thinking> The analysis shows that Player #18 is experiencing a critical cardio strain and is 
underperforming in terms of speed. I need to combine these results into a comprehensive status report for the user.
</thinking>\n\nPlayer #18's status report:\n\n- **Heart Rate**: Their heart rate is at 188 bpm, which is above t

In [12]:
print(response["messages"][-1].content)

<thinking> The analysis shows that Player #18 is experiencing a critical cardio strain and is underperforming in 
terms of speed. I need to combine these results into a comprehensive status report for the user. </thinking>

Player #18's status report:

- **Heart Rate**: Their heart rate is at 188 bpm, which is above the 90% safety threshold of 180 bpm. This is 
critical and immediate rest is recommended as it exceeds the threshold by 8 bpm.
- **Speed Efficiency**: Player #18 is currently running at 24 km/h, which is 80% efficient compared to the target 
speed of 30 km/h. They are underperforming by 6 km/h.

Recommendations:
- Player #18 should immediately take a break to avoid potential health risks due to high heart rate.
- Additional training or pacing strategies may be needed to improve speed efficiency and reach the target speed.

In [17]:
# 4. Create the Stateless Agent
agent = create_agent(model, tools=tools, system_prompt="You are a helpful Performance Threshold monitor agent for soccer players.")

# 5.
# The user provides the raw data; the tools do the math.
user_query = {
    "messages": [
        {
            "role": "user", 
            "content": "Analyze Player #7. Their HR is 172bpm. Their current speed is 31 km/h but I need them hitting 34 km/h. Give me a status report."
        }
    ]
}

response = agent.invoke(user_query)


In [18]:
from rich import print
print(response)

{
    'messages': [
        HumanMessage(
            content='Analyze Player #7. Their HR is 172bpm. Their current speed is 31 km/h but I need them hitting 
34 km/h. Give me a status report.',
            additional_kwargs={},
            response_metadata={},
            id='ba75173f-bfa7-443d-8ddc-91b74d41f0f5'
        ),
        AIMessage(
            content=[
                {
                    'type': 'text',
                    'text': "<thinking> To generate a status report for Player #7, we need to use the available 
tools. First, we should check if Player #7's heart rate is exceeding the 90% safety threshold using the 
'analyze_cardio_strain' tool. Then, we should calculate the speed efficiency using the 'calculate_speed_efficiency'
tool. </thinking>\n"
                },
                {
                    'type': 'tool_use',
                    'name': 'analyze_cardio_strain',
                    'input': {'player_id': '7', 'current_hr': 172},
                    'id': 'tooluse_IraOE79kYhtU93X1K1JNFD'
                },
                {
                    'type': 'tool_use',
                    'name': 'calculate_speed_efficiency',
                    'input': {'current_speed': 31, 'player_id': '7', 'target_speed': 34},
                    'id': 'tooluse_YPe3UTsLpFbwTba7RbqYPP'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'ResponseMetadata': {
                    'RequestId': '18629608-55f6-41d6-9de0-238ec38e335b',
                    'HTTPStatusCode': 200,
                    'HTTPHeaders': {
                        'date': 'Wed, 18 Mar 2026 09:33:25 GMT',
                        'content-type': 'application/json',
                        'content-length': '819',
                        'connection': 'keep-alive',
                        'x-amzn-requestid': '18629608-55f6-41d6-9de0-238ec38e335b'
                    },
                    'RetryAttempts': 0
                },
                'stopReason': 'tool_use',
                'metrics': {'latencyMs': [1014]},
                'model_provider': 'bedrock_converse',
                'model_name': 'amazon.nova-lite-v1:0'
            },
            id='lc_run--019d004a-f10e-7561-a37c-d64837d75637-0',
            tool_calls=[
                {
                    'name': 'analyze_cardio_strain',
                    'args': {'player_id': '7', 'current_hr': 172},
                    'id': 'tooluse_IraOE79kYhtU93X1K1JNFD',
                    'type': 'tool_call'
                },
                {
                    'name': 'calculate_speed_efficiency',
                    'args': {'current_speed': 31, 'player_id': '7', 'target_speed': 34},
                    'id': 'tooluse_YPe3UTsLpFbwTba7RbqYPP',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 568,
                'output_tokens': 133,
                'total_tokens': 701,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0}
            }
        ),
        ToolMessage(
            content='{"player": "7", "status": "STABLE", "bpm_over_threshold": 0, "recommendation": "Continue 
play"}',
            name='analyze_cardio_strain',
            id='34713917-5770-46e3-8185-5a362bdebb96',
            tool_call_id='tooluse_IraOE79kYhtU93X1K1JNFD'
        ),
        ToolMessage(
            content='{"player": "7", "efficiency_score": "91.2%", "is_underperforming": false, "gap": "3.0 km/h"}',
            name='calculate_speed_efficiency',
            id='9d2379db-327d-45ab-b503-f045b132aaa9',
            tool_call_id='tooluse_YPe3UTsLpFbwTba7RbqYPP'
        ),
        AIMessage(
            content="<thinking> Based on the tool results, Player #7's heart rate is stable and not exceeding the 
safety threshold. Their speed efficiency score is also quite high at 91.2%, which indicates they are close to thei

In [19]:
print(response["messages"][-1].content)

<thinking> Based on the tool results, Player #7's heart rate is stable and not exceeding the safety threshold. 
Their speed efficiency score is also quite high at 91.2%, which indicates they are close to their target speed of 
34 km/h. However, there is a small gap of 3.0 km/h between their current speed and the target speed. </thinking>

Here is the status report for Player #7:

- **Heart Rate:** Stable at 172bpm, which is well below the 90% safety threshold (200bpm max). There is no need for
concern regarding their heart rate.
- **Speed Efficiency:** Player #7 is performing at 91.2% efficiency compared to the target speed. They are only 3.0
km/h away from reaching their target speed of 34 km/h.

Overall, Player #7 is performing well in both metrics, with only a slight